In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# Replace with your folder path
users = pd.read_csv(r"D:\flatZ_data\users_realistic.csv")
items = pd.read_csv(r"D:\flatZ_data\items_realistic.csv")
interactions = pd.read_csv(r"D:\flatZ_data\interactions_realistic.csv")

print("Users:", users.shape)
print("Items:", items.shape)
print("Interactions:", interactions.shape)


Users: (100, 3)
Items: (100, 5)
Interactions: (1000, 5)


In [3]:
# Map interaction types to numeric weights
interactions['interaction'] = interactions['interaction_type'].map({
    'view': 1,
    'like': 2,
    'save': 3,
    'contact': 4
})

# Fill missing values with 1
interactions['interaction'] = interactions['interaction'].fillna(1)


In [4]:
user_item_matrix = interactions.pivot_table(
    index='user_id',
    columns='item_id',
    values='interaction',
    fill_value=0
)

user_item_np = user_item_matrix.to_numpy()


In [5]:
user_similarity = cosine_similarity(user_item_np)
np.fill_diagonal(user_similarity, 0)  # ignore self-similarity


In [6]:
train, test = train_test_split(interactions, test_size=0.2, random_state=42)

train_matrix = train.pivot_table(index='user_id', columns='item_id', values='interaction', fill_value=0)
test_matrix = test.pivot_table(index='user_id', columns='item_id', values='interaction', fill_value=0)

train_matrix = train_matrix.reindex(index=user_item_matrix.index, columns=user_item_matrix.columns, fill_value=0)
test_matrix = test_matrix.reindex(index=user_item_matrix.index, columns=user_item_matrix.columns, fill_value=0)


In [7]:
predicted_ratings = np.dot(user_similarity, train_matrix)


In [8]:
def recommend_for_user(user_id, top_n=5):
    user_index = user_item_matrix.index.get_loc(user_id)
    scores = predicted_ratings[user_index]
    
    # Ignore items already interacted
    interacted_items = train_matrix.loc[user_id]
    scores = np.where(interacted_items > 0, -1, scores)
    
    top_items = np.argsort(scores)[::-1][:top_n]
    recommended_ids = user_item_matrix.columns[top_items]
    
    return items[items['item_id'].isin(recommended_ids)]

# Example recommendation
print("Top 5 recommendations for user 1:")
print(recommend_for_user(1))


Top 5 recommendations for user 1:
    item_id                         title        city  price  \
2         3      3BHK Apartment in Mumbai      Mumbai  33065   
6         7        2BHK Apartment in Pune        Pune  49112   
44       45  3BHK Apartment in Coimbatore  Coimbatore  14820   
49       50        3BHK Apartment in Pune        Pune  37164   
52       53      1BHK Apartment in Jaipur      Jaipur  41270   

                                            amenities  
2   Power Backup, Play Area, Swimming Pool, Garden...  
6              Parking, Power Backup, Lift, Play Area  
44             Play Area, Power Backup, Lift, Parking  
49                          Parking, Security, Garden  
52      Gym, Security, Swimming Pool, Parking, Garden  


In [11]:
top_n = 5
precision_list = []

for uid in user_item_matrix.index:
    true_items = test_matrix.loc[uid]
    true_items = set(true_items[true_items > 0].index)
    
    recommended_items = recommend_for_user(uid, top_n=top_n)['item_id'].tolist()
    
    if recommended_items:
        precision = len(set(recommended_items) & true_items) / len(recommended_items)
        precision_list.append(precision)

overall_precision = np.mean(precision_list)
print("Overall Top-N Precision:", overall_precision)


Overall Top-N Precision: 0.196


In [12]:
def add_feedback(user_id, item_id, feedback_type='like'):
    feedback_weight = {'view': 1, 'like': 2, 'save': 3, 'contact': 4}
    value = feedback_weight.get(feedback_type, 1)
    
    global train_matrix, predicted_ratings
    if item_id in train_matrix.columns:
        train_matrix.at[user_id, item_id] = value
    else:
        train_matrix[item_id] = 0
        train_matrix.at[user_id, item_id] = value
    
    user_index = user_item_matrix.index.get_loc(user_id)
    predicted_ratings[user_index] = np.dot(user_similarity[user_index], train_matrix)

# Example
add_feedback(user_id=1, item_id=5, feedback_type='save')
print("Updated Top 5 recommendations for user 1 after feedback:")
print(recommend_for_user(1))


Updated Top 5 recommendations for user 1 after feedback:
    item_id                         title        city  price  \
2         3      3BHK Apartment in Mumbai      Mumbai  33065   
6         7        2BHK Apartment in Pune        Pune  49112   
44       45  3BHK Apartment in Coimbatore  Coimbatore  14820   
49       50        3BHK Apartment in Pune        Pune  37164   
52       53      1BHK Apartment in Jaipur      Jaipur  41270   

                                            amenities  
2   Power Backup, Play Area, Swimming Pool, Garden...  
6              Parking, Power Backup, Lift, Play Area  
44             Play Area, Power Backup, Lift, Parking  
49                          Parking, Security, Garden  
52      Gym, Security, Swimming Pool, Parking, Garden  


In [13]:
def recommend_cold_start(top_n=5):
    item_popularity = train_matrix.sum(axis=0)
    top_items = item_popularity.sort_values(ascending=False).head(top_n).index
    return items[items['item_id'].isin(top_items)]

print("Top 5 recommendations for a new cold-start user:")
print(recommend_cold_start(top_n=5))


Top 5 recommendations for a new cold-start user:
    item_id                         title        city  price  \
32       33       1BHK Apartment in Delhi       Delhi  46306   
39       40  1BHK Apartment in Coimbatore  Coimbatore  13967   
48       49       Luxury Villa in Chennai     Chennai  44391   
59       60             Penthouse in Pune        Pune   5974   
84       85     Studio Apartment in Delhi       Delhi  28429   

                                            amenities  
32  Garden, Play Area, Parking, Gym, Lift, Swimmin...  
39  Parking, Garden, Security, Play Area, Power Ba...  
48  Security, Lift, Garden, Parking, Play Area, Po...  
59  Lift, Swimming Pool, Power Backup, Security, P...  
84  Security, Gym, Play Area, Power Backup, Parkin...  


In [15]:
all_recommendations = []

for uid in user_item_matrix.index:
    top_items = recommend_for_user(uid, top_n=5)
    for item_id in top_items['item_id']:
        all_recommendations.append({
            'user_id': uid,
            'item_id': item_id
        })

# Convert to DataFrame
recommendations_df = pd.DataFrame(all_recommendations)

# Save as CSV
recommendations_df.to_csv(r"D:\flatZ_data\top5_recommendations.csv", index=False)
print("Top-5 recommendations for all users saved to 'top5_recommendations.csv'")

# Preview first 10 rows in notebook
recommendations_df.head(10)


Top-5 recommendations for all users saved to 'top5_recommendations.csv'


,user_id,item_id
0,1,3
1,1,7
2,1,45
3,1,50
4,1,53
5,2,27
6,2,60
7,2,72
8,2,74
9,2,95


In [17]:
# Show top-5 for user 1
recommendations_df[recommendations_df['user_id'] == 1]


,user_id,item_id
0,1,3
1,1,7
2,1,45
3,1,50
4,1,53


In [18]:
# Show top-5 for users 1,2,3
recommendations_df[recommendations_df['user_id'].isin([1,2,3])]


,user_id,item_id
0,1,3
1,1,7
2,1,45
3,1,50
4,1,53
5,2,27
6,2,60
7,2,72
8,2,74
9,2,95


In [19]:
recommendations_df.groupby('user_id').head(5)


,user_id,item_id
0,1,3
1,1,7
2,1,45
3,1,50
4,1,53
...,...,...
495,100,8
496,100,27
497,100,74
498,100,98


In [20]:
recommendations_df.to_csv(r"D:\flatZ_data\top5_recommendations.csv", index=False)


In [21]:
all_recommendations = []

for uid in user_item_matrix.index:
    top_items = recommend_for_user(uid, top_n=5)
    for item_id in top_items['item_id']:
        all_recommendations.append({
            'user_id': uid,
            'item_id': item_id
        })

# Convert to DataFrame
recommendations_df = pd.DataFrame(all_recommendations)

# Save as CSV
recommendations_df.to_csv(r"D:\flatZ_data\top5_recommendations.csv", index=False)
print("Top-5 recommendations for all users saved to 'top5_recommendations.csv'")

# Preview: show exactly 5 recommendations for each of the first 3 users
for uid in user_item_matrix.index[:3]:
    print(f"\nTop-5 recommendations for user {uid}:")
    display(recommendations_df[recommendations_df['user_id'] == uid])


Top-5 recommendations for all users saved to 'top5_recommendations.csv'

Top-5 recommendations for user 1:


,user_id,item_id
0,1,3
1,1,7
2,1,45
3,1,50
4,1,53



Top-5 recommendations for user 2:


,user_id,item_id
5,2,27
6,2,60
7,2,72
8,2,74
9,2,95



Top-5 recommendations for user 3:


,user_id,item_id
10,3,7
11,3,26
12,3,33
13,3,38
14,3,89
